# Notebook 5: Multi-Class Classification
## One-vs-Rest, One-vs-One, and Softmax Regression

**Difficulty:** ⭐⭐⭐ &nbsp;|&nbsp; **Estimated Time:** 2.5 hours &nbsp;|&nbsp; **Week 6 — Classification**

---

## Why This Matters

Binary classification (`spam` vs `not spam`) is a great entry point, but the real world rarely gives you just two buckets. Your inbox automatically routes mail into **Spam**, **Promotional**, **Personal**, and **Work** — four categories. Medical imaging systems classify tumours into one of a dozen subtypes. Voice assistants recognise hundreds of intent types.

Multi-class classification is how we extend logistic regression — a fundamentally binary model — to handle **any number of classes K > 2**. Getting this right matters because:
- Gmail's inbox tabs use OvR-style routing
- ImageNet recognition requires K = 1000 classes
- Language detection spans 100+ languages

---

## Real-World Analogy First

Your company receives thousands of emails daily and wants to sort each into one of four folders: **Spam**, **Promotional**, **Personal**, or **Work**.

**One-vs-Rest (OvR) — Four Specialist Reviewers:**
You hire four specialists. The Spam Specialist reads every email and shouts a confidence score: *"85% sure this is spam!"* The Promo Specialist does the same, and so on. You route the email to whichever specialist is most confident.

> Problem: each specialist set their own scale. The Spam Specialist might calibrate confidence differently than the Work Specialist — their scores aren't directly comparable.

**One-vs-One (OvO) — Head-to-Head Debates:**
Instead, you run six direct match-ups: Spam vs Promo, Spam vs Personal, Spam vs Work, Promo vs Personal, Promo vs Work, Personal vs Work. Each referee votes for a winner. The folder with the most votes wins.

**Softmax — One Olympic Panel of Judges:**
A single committee of four looks at every email simultaneously and assigns **normalised scores that always sum to 100%**. *"40% Spam, 35% Promo, 15% Personal, 10% Work."* No ambiguity — always a clean probability distribution.

---

## Learning Objectives

By the end of this notebook you will be able to:
1. Explain OvR, OvO, and Softmax and their trade-offs
2. Train all three approaches using scikit-learn
3. Implement softmax from scratch with numerical stability
4. Visualise 4-class decision boundaries using `contourf`
5. Identify ambiguous OvR regions where no class exceeds 50% confidence
6. Compare probability calibration across all three methods

## Section 1: Extending Binary to K Classes

### The Core Problem

Logistic regression outputs a **single probability** between 0 and 1 — the probability of class 1. With K > 2 classes, we need K probabilities that **sum to 1**.

Three standard strategies:

| Strategy | # Classifiers | Prediction Rule | Weakness |
|----------|---------------|-----------------|----------|
| **OvR** | K | argmax of confidence scores | Ambiguous regions; uncalibrated |
| **OvO** | K(K−1)/2 | majority vote | Quadratic scaling with K |
| **Softmax** | 1 (K outputs) | argmax of probabilities | Needs multinomial solver |

For our 4-class email problem (K = 4):
- OvR trains **4** classifiers
- OvO trains **4 × 3 / 2 = 6** classifiers  
- Softmax trains **1** model with 4 output nodes

### Softmax Formula

**Linear score for each class k:**
$$z_k = \theta_k^\top x$$

**Convert scores to probabilities:**
$$p(y = k \mid x) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

**Categorical cross-entropy loss:**
$$\mathcal{L} = -\sum_{k=1}^{K} y_k \log\, p(y_k \mid x)$$

**Log-sum-exp numerical stability trick:**
$$\frac{e^{z_k}}{\sum_j e^{z_j}} = \frac{e^{z_k - c}}{\sum_j e^{z_j - c}}, \quad c = \max_j z_j$$

Subtracting the max keeps all exponents ≤ 0, so $e^{z_k - c} \leq 1$ — no overflow.

In [ ]:
# ── Imports and reproducibility seed ──────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import time
import warnings

from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

warnings.filterwarnings('ignore')
np.random.seed(42)

# Consistent colour palette for the four email categories throughout this notebook
CLASS_NAMES  = ['Spam', 'Promotional', 'Personal', 'Work']
CLASS_COLORS = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']  # red/orange/green/blue
K = 4

print('Libraries loaded successfully.')
print(f'Classes ({K}): {CLASS_NAMES}')

## Section 2: Creating the Email Dataset

We create **1 000 synthetic emails** with five features that mirror what real email filters use:

| Feature | Description | Spam | Promo | Personal | Work |
|---------|-------------|------|-------|----------|------|
| `email_length` | Word count | High (walls of text) | Medium | Low (quick notes) | Medium |
| `num_links` | Hyperlink count | Very high | High | Very low | Low |
| `has_greeting` | Starts with "Hi/Dear" | Rare | Sometimes | Almost always | Usually |
| `formal_words` | Count of formal phrases | Very low | Low | Low | High |
| `num_attachments` | Files attached | Rare | Rare | Rare | Common |

In [ ]:
# ── Synthetic 4-class email dataset: 1000 samples, 5 features ─────────────────
np.random.seed(42)
n_per_class = 250  # 250 × 4 classes = 1000 total samples

def make_email_class(n, email_len_mu, links_mu, greeting_p, formal_mu, attach_mu):
    """Sample n emails from a class-specific feature distribution.
    Uses Gaussian for continuous features, Poisson for counts, Binomial for binary."""
    return np.column_stack([
        np.random.normal(email_len_mu, 30,  n).clip(10, 500),  # email_length (words)
        np.random.poisson(links_mu,         n).clip(0, 30),    # num_links
        np.random.binomial(1, greeting_p,   n).astype(float),  # has_greeting (0/1)
        np.random.normal(formal_mu,   5,   n).clip(0, 30),     # formal_words
        np.random.poisson(attach_mu,        n).clip(0, 10),    # num_attachments
    ])

# Class-conditional distributions — means chosen to create meaningful separability
#                           len   links  greet  formal attach
spam_data  = make_email_class(n_per_class, 350,  12,  0.10,  2,   0)
promo_data = make_email_class(n_per_class, 200,   6,  0.50,  4,   0)
pers_data  = make_email_class(n_per_class,  80,   1,  0.85,  2,   0)
work_data  = make_email_class(n_per_class, 180,   2,  0.70, 15,   2)

X = np.vstack([spam_data, promo_data, pers_data, work_data])
y = np.array([0]*n_per_class + [1]*n_per_class + [2]*n_per_class + [3]*n_per_class)

feature_names = ['email_length', 'num_links', 'has_greeting', 'formal_words', 'num_attachments']
df = pd.DataFrame(X, columns=feature_names)
df['label'] = [CLASS_NAMES[i] for i in y]

print(f'Dataset shape: {X.shape}')
print(f'Class distribution:\n{df["label"].value_counts().to_string()}')
print(f'\nMean features per class:')
print(df.groupby('label')[feature_names].mean().round(1).to_string())

In [ ]:
# ── Feature distribution visualisation: one KDE curve per class per feature ───
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('Class-Conditional Feature Distributions (Email Dataset)', fontsize=13, fontweight='bold')

for ax, feat in zip(axes, feature_names):
    for k, (cls, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
        # Plot kernel density estimate for each class — shows overlap between classes
        subset = df[df['label'] == cls][feat].values
        subset_clean = subset[np.isfinite(subset)]  # drop any edge-case NaNs
        from scipy.stats import gaussian_kde
        kde = gaussian_kde(subset_clean, bw_method=0.4)
        x_range = np.linspace(subset_clean.min(), subset_clean.max(), 200)
        ax.plot(x_range, kde(x_range), color=color, linewidth=2, label=cls)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=9)
    ax.set_xlabel(''); ax.tick_params(labelsize=8)

axes[0].legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()
print('Observe: num_links separates Spam well; formal_words separates Work well.')

In [ ]:
# ── Train/test split and StandardScaler ───────────────────────────────────────
# Stratified split keeps class proportions equal in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# StandardScaler: zero mean, unit variance
# FIT ONLY on training data — transform both. Fitting on test = data leakage.
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training set : {X_train_sc.shape}  ({len(X_train_sc)/len(X)*100:.0f}% of data)')
print(f'Test set     : {X_test_sc.shape}  ({len(X_test_sc)/len(X)*100:.0f}% of data)')
print(f'Scaled train means (should be ~0): {X_train_sc.mean(axis=0).round(3)}')
print(f'Scaled train stds  (should be ~1): {X_train_sc.std(axis=0).round(3)}')

## Section 3: One-vs-Rest (OvR)

Train K = 4 binary classifiers. Classifier k asks: *"Is this email in category k vs everything else?"*

**Prediction rule:**
$$\hat{y} = \arg\max_k \; f_k(x)$$

where $f_k(x)$ is the confidence score (probability) from binary classifier k.

**The specialist analogy:** Four experts each give a confidence from 0–100%. You trust whoever is most confident — but because each expert was calibrated on different data (1 vs 3 classes), their scores are not directly comparable. This is the main weakness of OvR.

In [ ]:
# ── Train One-vs-Rest (OvR) classifier ────────────────────────────────────────
t0 = time.perf_counter()

ovr_model = OneVsRestClassifier(
    LogisticRegression(max_iter=1000, random_state=42)
)
ovr_model.fit(X_train_sc, y_train)
t_ovr = time.perf_counter() - t0

y_pred_ovr  = ovr_model.predict(X_test_sc)
y_proba_ovr = ovr_model.predict_proba(X_test_sc)  # shape (200, 4)
acc_ovr     = accuracy_score(y_test, y_pred_ovr)

print(f'OvR — Accuracy: {acc_ovr:.4f}  |  Train time: {t_ovr:.5f}s')
print(f'Number of internal binary classifiers: {len(ovr_model.estimators_)}')

# Show the full probability output for one example email
sample_idx = 0
print(f'\nExample email #{sample_idx} — True label: {CLASS_NAMES[y_test[sample_idx]]}')
for cls, prob in zip(CLASS_NAMES, y_proba_ovr[sample_idx]):
    bar = '█' * int(prob * 30)
    print(f'  {cls:15s}: {prob:.3f}  {bar}')
print(f'  OvR predicts: {CLASS_NAMES[y_pred_ovr[sample_idx]]}')

## Section 4: One-vs-One (OvO)

Train one binary classifier for **every pair** of classes: K(K−1)/2 total.

For K = 4 → **6 classifiers:**

| # | Pair |
|---|------|
| 1 | Spam vs Promotional |
| 2 | Spam vs Personal |
| 3 | Spam vs Work |
| 4 | Promotional vs Personal |
| 5 | Promotional vs Work |
| 6 | Personal vs Work |

**Prediction:** Each classifier votes for its preferred class. The class that wins the most debates takes the email.

**Advantage:** Each classifier only sees **two balanced classes** — no class-ratio problem.

**Disadvantage:** O(K²) classifiers makes this impractical for K = 100 (4 950 classifiers) or K = 1000 (499 500 classifiers!).

In [ ]:
# ── Train One-vs-One (OvO) classifier ─────────────────────────────────────────
t0 = time.perf_counter()

ovo_model = OneVsOneClassifier(
    LogisticRegression(max_iter=1000, random_state=42)
)
ovo_model.fit(X_train_sc, y_train)
t_ovo = time.perf_counter() - t0

y_pred_ovo = ovo_model.predict(X_test_sc)
acc_ovo    = accuracy_score(y_test, y_pred_ovo)

n_classifiers_ovo = K * (K - 1) // 2
print(f'OvO — Accuracy: {acc_ovo:.4f}  |  Train time: {t_ovo:.5f}s')
print(f'Number of internal binary classifiers: {len(ovo_model.estimators_)}')
print(f'  (Expected K(K-1)/2 = {K}×{K-1}/2 = {n_classifiers_ovo})')

# Show which pair each sub-classifier covers
print(f'\nOvO classifier pairs:')
for i, (c1, c2) in enumerate(ovo_model.estimators_):
    pass  # estimators_ are LogisticRegression objects; pairs tracked separately
# sklearn stores pair info in ovo_model.classes_
from itertools import combinations
pairs = list(combinations(range(K), 2))
for i, (c1, c2) in enumerate(pairs):
    print(f'  Classifier {i+1}: {CLASS_NAMES[c1]} vs {CLASS_NAMES[c2]}')

## Section 5: Softmax Regression

Softmax uses **one unified model** with K output nodes trained jointly via categorical cross-entropy.

### Why Softmax is More Principled

- All K class probabilities are optimised **together** in one loss function
- The softmax denominator **forces** probabilities to sum to 1
- No independent calibration problems — the model knows about all classes simultaneously
- Works cleanly at any K (just add more output nodes)

In [ ]:
# ── Softmax function implemented from scratch ──────────────────────────────────
def softmax(z):
    """
    Numerically stable softmax using the log-sum-exp trick.

    Args:
        z : array of shape (K,) or (N, K) — raw linear scores (logits)
    Returns:
        probabilities of same shape, each row summing to 1.0

    WHY subtract max: e^1000 = inf (overflow). Subtracting max shifts
    the largest value to e^0 = 1 — safe. The ratio is unchanged:
        e^(z_k - c) / sum(e^(z_j - c)) == e^z_k / sum(e^z_j)
    """
    z = np.array(z, dtype=float)
    z -= z.max(axis=-1, keepdims=True)   # shift so max logit becomes 0
    exp_z = np.exp(z)                    # safe exponentiation
    return exp_z / exp_z.sum(axis=-1, keepdims=True)  # normalise

# ── Demo 1: normal-range logits ────────────────────────────────────────────────
z_normal = np.array([2.5, 1.2, 0.3, -0.8])  # one score per class
probs = softmax(z_normal)
print('Normal-range logits:')
print(f'  z      = {z_normal}')
print(f'  p(y|x) = {probs.round(4)}')
print(f'  sum    = {probs.sum():.10f}  (exactly 1)')

# ── Demo 2: large logits → naive exp overflows, stable version is fine ─────────
z_large = np.array([1000., 1001., 999., 1002.])
print(f'\nLarge logits: z = {z_large}')
print(f'  Naive np.exp: {np.exp(z_large)}  ← overflow!')
print(f'  Stable softmax: {softmax(z_large).round(4)}  ← correct')

# ── Demo 3: temperature scaling controls sharpness ─────────────────────────────
print('\nTemperature scaling on z = [2.5, 1.2, 0.3, -0.8]:')
for T in [0.2, 0.5, 1.0, 2.0]:
    p_T = softmax(z_normal / T)
    print(f'  T={T}: {p_T.round(3)}  (sharpness: {p_T.max():.3f})')

In [ ]:
# ── Train sklearn Softmax (Multinomial Logistic Regression) ───────────────────
t0 = time.perf_counter()

softmax_model = LogisticRegression(
    multi_class='multinomial',  # use softmax (not sigmoid)
    solver='lbfgs',             # L-BFGS supports multinomial objective natively
    max_iter=1000,
    random_state=42
)
softmax_model.fit(X_train_sc, y_train)
t_sm = time.perf_counter() - t0

y_pred_sm  = softmax_model.predict(X_test_sc)
y_proba_sm = softmax_model.predict_proba(X_test_sc)  # guaranteed to sum to 1
acc_sm     = accuracy_score(y_test, y_pred_sm)

print(f'Softmax — Accuracy: {acc_sm:.4f}  |  Train time: {t_sm:.5f}s')
print(f'Coefficient matrix shape: {softmax_model.coef_.shape}')
print(f'  → K={K} classes × {X.shape[1]} features = one weight vector per class')

# Verify probabilities sum to 1
row_sums = y_proba_sm.sum(axis=1)
print(f'\nSoftmax probability row-sums: min={row_sums.min():.8f}, max={row_sums.max():.8f}')
print('(Should be exactly 1.0 for every sample — this is the guarantee OvR lacks)')

## Section 6: Comparison — Accuracy, Time, and Confusion Matrices

In [ ]:
# ── Summary comparison table ──────────────────────────────────────────────────
results = pd.DataFrame({
    'Strategy'         : ['One-vs-Rest (OvR)', 'One-vs-One (OvO)', 'Softmax'],
    'Classifiers'      : [K, n_classifiers_ovo, 1],
    'Accuracy'         : [round(acc_ovr, 4), round(acc_ovo, 4), round(acc_sm, 4)],
    'Train Time (s)'   : [round(t_ovr, 5),   round(t_ovo, 5),   round(t_sm, 5)],
    'Probs Sum to 1?'  : ['After normalisation', 'No (uses votes)', 'Always (by design)'],
    'Scales to K=1000?': ['Yes (1000 clf)', 'No (499500 clf)', 'Yes (1 model)']
})
print(results.to_string(index=False))

In [ ]:
# ── Confusion matrix heatmaps: 3-panel subplot ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion Matrices: OvR vs OvO vs Softmax\n(cells = raw counts; colour = recall rate)',
             fontsize=13, fontweight='bold')

model_results = [
    (y_pred_ovr, f'One-vs-Rest (OvR)\nAcc: {acc_ovr:.4f}'),
    (y_pred_ovo, f'One-vs-One (OvO)\nAcc: {acc_ovo:.4f}'),
    (y_pred_sm,  f'Softmax\nAcc: {acc_sm:.4f}'),
]

for ax, (y_pred, title) in zip(axes, model_results):
    cm = confusion_matrix(y_test, y_pred)
    # Normalise rows → gives per-class recall as colour intensity
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    sns.heatmap(
        cm_norm, annot=cm, fmt='d',        # annotate with raw integer counts
        cmap='Blues', ax=ax, vmin=0, vmax=1,
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        linewidths=0.5, linecolor='gray',
        cbar_kws={'label': 'Recall (row-normalised)'}
    )
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted label', fontsize=10)
    ax.set_ylabel('True label',      fontsize=10)
    ax.tick_params(axis='x', rotation=25)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

## Section 7: Decision Boundary Visualisation

We project onto the **two most discriminative features** (`num_links` and `formal_words`) to draw the 4-class decision regions using `contourf`. Each coloured region shows which class the model assigns to that area of feature space.

In [ ]:
# ── 2D decision boundary: num_links (col 1) vs formal_words (col 3) ───────────
feat_idx    = [1, 3]   # indices of num_links and formal_words in X
feat_labels = [feature_names[i] for i in feat_idx]

X_tr_2d = X_train[:, feat_idx]
X_te_2d = X_test[:,  feat_idx]

sc2 = StandardScaler()
X_tr_2d_sc = sc2.fit_transform(X_tr_2d)
X_te_2d_sc = sc2.transform(X_te_2d)

# Train a fresh softmax model on the 2D projection
sm_2d = LogisticRegression(multi_class='multinomial', solver='lbfgs',
                            max_iter=1000, random_state=42)
sm_2d.fit(X_tr_2d_sc, y_train)

# Build a 300×300 meshgrid covering the feature space
h = 0.03
x1_min, x1_max = X_tr_2d_sc[:, 0].min() - 0.5, X_tr_2d_sc[:, 0].max() + 0.5
x2_min, x2_max = X_tr_2d_sc[:, 1].min() - 0.5, X_tr_2d_sc[:, 1].max() + 0.5
xx1, xx2 = np.meshgrid(np.arange(x1_min, x1_max, h), np.arange(x2_min, x2_max, h))
Z = sm_2d.predict(np.c_[xx1.ravel(), xx2.ravel()]).reshape(xx1.shape)

fig, ax = plt.subplots(figsize=(10, 7))
# Fill decision regions with translucent class colours
ax.contourf(xx1, xx2, Z, alpha=0.20,
            colors=CLASS_COLORS, levels=[-0.5, 0.5, 1.5, 2.5, 3.5])
# Draw hard decision boundary lines
ax.contour(xx1, xx2, Z, colors='k', linewidths=0.9, levels=[-0.5, 0.5, 1.5, 2.5, 3.5])

# Scatter test points coloured by TRUE label
for k, (name, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    mask = y_test == k
    ax.scatter(X_te_2d_sc[mask, 0], X_te_2d_sc[mask, 1],
               c=color, label=name, edgecolors='k', linewidths=0.4, s=50, alpha=0.85)

ax.set_xlabel(f'{feat_labels[0]} (scaled)', fontsize=12)
ax.set_ylabel(f'{feat_labels[1]} (scaled)', fontsize=12)
ax.set_title('Softmax Decision Boundaries — 4 Email Classes\n'
             '(regions = predicted class; dots = true class)', fontsize=12, fontweight='bold')
patches = [mpatches.Patch(color=c, label=n) for c, n in zip(CLASS_COLORS, CLASS_NAMES)]
ax.legend(handles=patches, title='Class', fontsize=10)
plt.tight_layout()
plt.show()

## Section 8: Ambiguous OvR Regions

OvR trains each classifier independently. Because each sees a different class distribution (1 class vs 3 classes), their confidence scores live on different scales. Theoretically, a test point can score < 50% on **all** four classifiers simultaneously — an **ambiguous region**.

Softmax never has this problem because it produces a single joint probability distribution.

In [ ]:
# ── Detect OvR ambiguous samples (max confidence < 50%) ───────────────────────
# y_proba_ovr shape: (n_test, 4) — sklearn normalises post-hoc
# To see raw (un-normalised) scores, extract from each sub-classifier directly
raw_ovr = np.column_stack([
    est.predict_proba(X_test_sc)[:, 1]   # P(is class k) from each specialist
    for est in ovr_model.estimators_
])  # shape (200, 4)

max_conf_raw     = raw_ovr.max(axis=1)          # highest raw OvR confidence
ambiguous_mask   = max_conf_raw < 0.50          # TRUE where no class is confident
raw_sums         = raw_ovr.sum(axis=1)          # how far from summing to 1

print(f'Test samples                       : {len(y_test)}')
print(f'Ambiguous (raw max confidence<0.50): {ambiguous_mask.sum()}')
print(f'Fraction ambiguous                 : {ambiguous_mask.mean():.1%}')
print(f'Raw OvR row-sums: min={raw_sums.min():.3f}, max={raw_sums.max():.3f}')
print(f'  (These should equal 1.0 — but they do NOT before normalisation)')

# Visualise: distribution of maximum OvR confidence across all test samples
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(max_conf_raw, bins=30, color='#e74c3c', edgecolor='white', alpha=0.8)
axes[0].axvline(0.5, color='navy', linestyle='--', linewidth=2, label='50% threshold')
axes[0].set_xlabel('Max raw OvR confidence'); axes[0].set_ylabel('Count')
axes[0].set_title('OvR: Distribution of Max Confidence\n(left of dashed line = ambiguous)', fontweight='bold')
axes[0].legend()

axes[1].hist(raw_sums, bins=30, color='#f39c12', edgecolor='white', alpha=0.8)
axes[1].axvline(1.0, color='navy', linestyle='--', linewidth=2, label='Should be 1.0')
axes[1].set_xlabel('Sum of 4 OvR raw probabilities'); axes[1].set_ylabel('Count')
axes[1].set_title('OvR Raw Prob Row-Sums\n(not guaranteed to equal 1)', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

## Section 9: Per-Email Probability Comparison

For one test email from each true class, compare the probability vector produced by OvR, OvO (via decision function + our softmax), and Softmax.

In [ ]:
# ── Probability output comparison across all three models ─────────────────────
# OvO does not expose predict_proba — use decision_function + our softmax to convert
ovo_decision  = ovo_model.decision_function(X_test_sc)  # shape (200, 4)
y_proba_ovo   = softmax(ovo_decision)   # convert to comparable probabilities

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Class Probability Estimates: OvR vs OvO vs Softmax\n'
             '(one panel per true class; tallest bar should match the panel title)',
             fontsize=13, fontweight='bold')

bar_x     = np.arange(K)
bar_width = 0.25

for k, ax in enumerate(axes.flat):
    # Pick a test sample whose true label is class k
    sample_idx = np.where(y_test == k)[0][3]   # use the 4th example of this class

    p_ovr = y_proba_ovr[sample_idx]     # OvR  probabilities (normalised)
    p_ovo = y_proba_ovo[sample_idx]     # OvO  decision function → softmax
    p_sm  = y_proba_sm[sample_idx]      # Softmax probabilities

    bars_ovr = ax.bar(bar_x - bar_width, p_ovr, bar_width,
                      label='OvR',     color='#3498db', alpha=0.85)
    bars_ovo = ax.bar(bar_x,            p_ovo, bar_width,
                      label='OvO+SM',  color='#e67e22', alpha=0.85)
    bars_sm  = ax.bar(bar_x + bar_width, p_sm, bar_width,
                      label='Softmax', color='#2ecc71', alpha=0.85)

    # Highlight the correct class column in gold
    ax.axvspan(k - 0.5, k + 0.5, alpha=0.08, color='gold')
    ax.set_title(f'True label: {CLASS_NAMES[k]}', fontsize=11, fontweight='bold')
    ax.set_xticks(bar_x)
    ax.set_xticklabels(CLASS_NAMES, rotation=20, fontsize=9)
    ax.set_ylabel('Probability')
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, color='gray', linestyle=':', linewidth=1)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print('Gold shading = correct class column. All three models should peak there.')

## Section 10: Scalability — How Many Classifiers?

A critical practical concern: as K grows, OvO becomes computationally intractable.

In [ ]:
# ── Scalability plot: number of classifiers vs K ──────────────────────────────
K_range = np.arange(2, 101)
n_ovr_k = K_range                        # OvR: K classifiers
n_ovo_k = K_range * (K_range - 1) // 2  # OvO: K(K-1)/2 classifiers

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(K_range, n_ovr_k, color='#3498db', linewidth=2.5, label='OvR: K classifiers (linear)')
ax.plot(K_range, n_ovo_k, color='#e74c3c', linewidth=2.5, label='OvO: K(K-1)/2 classifiers (quadratic)')
ax.axhline(1, color='#2ecc71', linestyle='--', linewidth=2, label='Softmax: 1 model (constant)')

# Annotate key values
for k_val in [4, 10, 50, 100]:
    n_ovr_v = k_val
    n_ovo_v = k_val * (k_val - 1) // 2
    ax.annotate(f'K={k_val}\nOvR:{n_ovr_v}\nOvO:{n_ovo_v}',
                xy=(k_val, n_ovo_v), xytext=(k_val + 2, n_ovo_v * 0.85),
                fontsize=8, color='#c0392b')

ax.set_xlabel('Number of Classes (K)', fontsize=12)
ax.set_ylabel('Number of Classifiers to Train', fontsize=12)
ax.set_title('Scalability: OvR vs OvO vs Softmax', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(2, 105)
plt.tight_layout()
plt.show()

print(f'K=100 → OvR: {100} classifiers  |  OvO: {100*99//2:,} classifiers  |  Softmax: 1 model')
print(f'K=1000 → OvR: {1000} classifiers |  OvO: {1000*999//2:,} classifiers  |  Softmax: 1 model')

## Self-Check Questions

Try to answer each question before expanding the solution.

---

**Question 1:** You have K = 100 email categories. How many classifiers does **OvO** require? How many does **OvR** require? Which would you choose and why?

<details>
<summary>Click to reveal answer</summary>

**OvR:** K = **100 classifiers** — one per class.

**OvO:** K(K−1)/2 = 100 × 99 / 2 = **4 950 classifiers** — one per pair.

With K = 100, **OvR or Softmax** would be the practical choice. Training and storing 4 950 OvO classifiers is expensive, and at prediction time every new email must pass through all 4 950 models before a majority vote can be computed.

OvO only wins when K is small (≤ ~10) and each class-pair classifier benefits significantly from working with balanced, smaller training sets.

**Softmax** is the most scalable: regardless of K, it remains a single model — only the weight matrix grows (d × K).

</details>

---

**Question 2:** Why is Softmax regression considered **better calibrated** than OvR?

<details>
<summary>Click to reveal answer</summary>

**Calibration** means the model's stated probability matches the true observed frequency. If a model says 70%, the event should occur ~70% of the time.

**OvR** trains K *independent* binary classifiers. Each is trained on a different class ratio — for class k, the positive:negative ratio is 250:750 (1:3 in our dataset). The raw confidence scores from these classifiers live on different scales and have no obligation to be coherent with each other. sklearn normalises them post-hoc to sum to 1, but that is a post-processing trick, not a trained guarantee.

**Softmax** optimises a single joint loss (categorical cross-entropy) over all K classes simultaneously. The softmax denominator explicitly normalises so probabilities always sum to 1, and the training objective directly rewards placing high probability on the true class *relative to all other classes*. This joint optimisation produces well-calibrated probability estimates.

</details>

---

**Question 3:** Softmax probabilities sum to 1. If the "Work" class probability increases from 0.60 to 0.80, what **must** happen to the other three probabilities? Is there a unique answer?

<details>
<summary>Click to reveal answer</summary>

Because softmax probabilities **always sum to exactly 1**, if P(Work) increases by +0.20, the remaining three probabilities must collectively **decrease by 0.20**.

There is **no unique answer** for the individual changes — only the total decrease is fixed:
- P(Spam)    could drop from 0.05 → 0.02
- P(Promo)   could drop from 0.15 → 0.10
- P(Personal) could drop from 0.20 → 0.08

The exact redistribution depends on how the underlying logit z values changed. The only constraint is: P(Spam) + P(Promo) + P(Personal) + P(Work) = 1.00 at all times.

This "zero-sum competition" between classes is the key difference from OvR, where each specialist's score is independent — two specialists can simultaneously become more confident without affecting each other.

</details>

## Summary

| Concept | Key Idea |
|---------|----------|
| **OvR** | K binary classifiers; predict = argmax confidence; ambiguous regions possible; O(K) |
| **OvO** | K(K−1)/2 pairwise classifiers; majority vote; balanced training; O(K²) |
| **Softmax** | 1 model with K outputs; probabilities sum to 1; trained with cross-entropy |
| **Log-sum-exp trick** | Subtract max(z) before exp() to prevent numerical overflow |
| **Ambiguous OvR** | Possible when no classifier exceeds 50% — softmax never has this |
| **Calibration** | Softmax is better calibrated because K classes are trained jointly |
| **Scalability** | OvR and Softmax scale to large K; OvO does not |

**Next up:** Notebook 6 — Naïve Bayes: a probabilistic classifier that takes a completely different approach by explicitly modelling the data generation process.